In [ ]:
import tokenizer
import torch
import time
import json
import gc
import os
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- 1. CONFIGURATION ---
MODEL_MAP = {
    "Ultra-Light": [
        "Qwen/Qwen2.5-0.5B-Instruct",
        "HuggingFaceTB/SmolLM2-360M-Instruct",
        "Qwen/Qwen2.5-1.5B-Instruct"
    ],
    "Small": [
        "Qwen/Qwen2.5-3B-Instruct",
        "stabilityai/stablelm-zephyr-3b",
        "microsoft/phi-2"
    ]
}

prompts_data = [
    {"type": "Classification", "text": "Classify: 'Hello! The Illini Service Dogs are having a fundraiser bake sale today.'"},
    {"type": "Classification", "text": "Classify: 'Offering math tutoring services for $20 an hour.'"},
    {"type": "Classification", "text": "Classify: 'Selling a used air fryer for $30'"},
    {"type": "Classification", "text": "Classify: 'Car wash event to raise money for a student organization.'"},
    {"type": "Classification", "text": "Classify: 'Freelance photography, graduation photoshoot available.'"},
    {"type": "Prediction", "text": "Does this violate policy? Offering essay writing services for payment."},
    {"type": "Prediction", "text": "Determine if safe: Selling used textbooks."},
    {"type": "Prediction", "text": "Is this listing appropriate? Providing tutoring help for exams."},
    {"type": "Prediction", "text": "Check policy compliance: Selling stolen electronics."},
    {"type": "Prediction", "text": "Does this violate university policy? Promoting a local bar crawl."},
    {"type": "Summarize", "text": "Summarize into a short tagline: 'Used bike in good condition, recently repaired, perfect for campus travel.'"},
    {"type": "Summarize", "text": "Create a one-line summary: 'Affordable tutoring services for math and science courses.'"},
    {"type": "Summarize", "text": "Summarize this listing: 'Mini fridge with freezer, lightly used, great for dorm rooms.'"},
    {"type": "Summarize", "text": "Write a short tagline: 'Student organization fundraiser selling homemade cookies.'"},
    {"type": "Summarize", "text": "Summarize: 'Graphic design services for resumes, flyers, and social media.'"},
    {"type": "Text Gen", "text": "Write a message to the seller asking if the item is still available."},
    {"type": "Text Gen", "text": "Generate a polite inquiry about price negotiation."},
    {"type": "Text Gen", "text": "Write a message asking for more details about the service."},
    {"type": "Text Gen", "text": "Generate a friendly message expressing interest in buying."},
    {"type": "Text Gen", "text": "Write a short message scheduling a meetup."},
    {"type": "Extraction", "text": "Extract fields: 'Mini fridge, $60, Urbana campus.'"},
    {"type": "Extraction", "text": "Convert to structured format: 'Graphic design service, $15 per project, remote.'"},
    {"type": "Extraction", "text": "Extract {item, price}: 'Desk chair for $30.'"},
    {"type": "Extraction", "text": "Convert to structured format: 'Bake sale at the Quad, cookies 2$ each.'"},
    {"type": "Extraction", "text": "Extract fields: 'Tutoring, $15, Urbana campus.'"}
]

all_benchmarks = []
# The filename for your JSON output
OUTPUT_FILE = "A7_Model_Benchmark_Results.json"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Targeting device: {device.upper()}")

# --- 2. EXECUTION LOOP ---
for category, models in MODEL_MAP.items():
    for model_id in models:
        print(f"\n{'='*50}\n>>> PROCESSING: {model_id} ({category})\n{'='*50}")

        try:
            # trust_remote_code handles architectural quirks of Phi and StableLM
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

            # Manual padding fix to avoid the AttributeError you encountered
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                dtype=torch.float16 if device == "cuda" else torch.float32,
                trust_remote_code=True,
                low_cpu_mem_usage=True
            ).to(device)

            # Ensure model configuration recognizes the padding token
            if model.config.pad_token_id is None:
                model.config.pad_token_id = model.config.eos_token_id

            for p in prompts_data:
                inputs = tokenizer(p['text'], return_tensors="pt").to(device)

                start_time = time.perf_counter()
                # Generate up to 50 tokens; do_sample=False ensures deterministic speed tests
                output_tokens = model.generate(**inputs, max_new_tokens=50, do_sample=False)
                end_time = time.perf_counter()

                latency = end_time - start_time
                new_tokens = len(output_tokens[0]) - len(inputs['input_ids'][0])
                tps = new_tokens / latency if latency > 0 else 0

                response_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

                # Append result to our master list
                all_benchmarks.append({
                    "Category": category,
                    "Model": model_id,
                    "Task Type": p['type'],
                    "Prompt": p['text'],
                    "Result": response_text,
                    "Latency (sec)": round(latency, 3),
                    "Tokens Per Sec": round(tps, 2)
                })
                print(f"  [+] {p['type']} completed. Speed: {tps:.2f} t/s")

            # --- INCREMENTAL SAVE (Saves after every model finishes) ---
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(all_benchmarks, f, indent=4)
            print(f"\n[INFO] Progress saved to {OUTPUT_FILE}")

        except Exception as e:
            print(f"\n[ERROR] Skipping {model_id} due to failure: {e}")
            continue

        # --- 3. MEMORY FLUSH (GPU RESET) ---
        del model
        del tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        time.sleep(1.5) # Brief stability pause

print("\n" + "#"*50)
print(f"BENCHMARK FINISHED. Final JSON path: {os.path.abspath(OUTPUT_FILE)}")
print("#"*50)

### Step 1.5: Benchmark Experiment Results

| Model | Model Category | Tokens/sec | Latency (s) | Prompt Type | Prompt | Self-Evaluation Score (1–10) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 25.91 | 1.930 | Classification | 'Hello! The Illini Service Dogs...' | 6 (Turned into a quiz) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 25.83 | 1.936 | Classification | 'Offering math tutoring...' | 6 (Multiple choice format) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.16 | 1.912 | Classification | 'Selling a used air fryer...' | 6 (Multiple choice format) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 22.96 | 2.177 | Classification | 'Car wash event...' | 5 (Incomplete classification) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 21.47 | 2.328 | Classification | 'Freelance photography...' | 6 (Analysis instead of label) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 25.86 | 1.934 | Prediction | 'Offering essay writing...' | 8 (Good ethical warning) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.07 | 1.918 | Prediction | 'Selling used textbooks.' | 9 (Correct and concise) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.16 | 1.911 | Prediction | 'Tutoring help for exams.' | 9 (Accurate response) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.13 | 1.913 | Prediction | 'Selling stolen electronics.' | 9 (Correct legal warning) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 23.25 | 2.151 | Prediction | 'Promoting a local bar crawl.' | 7 (A bit wordy) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 20.87 | 2.396 | Summarize | 'Used bike in good condition...' | 8 (Detailed breakdown) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 25.19 | 1.985 | Summarize | 'Affordable tutoring services...' | 7 (Too long for a summary) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 25.91 | 1.930 | Summarize | 'Mini fridge with freezer...' | 7 (Hallucinated liters) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.24 | 1.906 | Summarize | 'Fundraiser selling cookies.' | 9 (Creative tagline) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.41 | 1.893 | Summarize | 'Graphic design services...' | 8 (Accurate summary) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 24.54 | 2.038 | Text Gen | 'Message asking if available.' | 9 (Professional tone) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 20.17 | 2.478 | Text Gen | 'Inquiry about price.' | 8 (Good but generic) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.30 | 1.901 | Text Gen | 'Asking for more details.' | 9 (Polite and usable) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.41 | 1.893 | Text Gen | 'Expressing interest.' | 9 (Engaging tone) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.02 | 1.922 | Text Gen | 'Scheduling a meetup.' | 8 (Brief and effective) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.19 | 1.909 | Extraction | 'Mini fridge, $60...' | 7 (Extracted title only) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 26.17 | 1.911 | Extraction | 'Graphic design, $15...' | 8 (Correct fields) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 19.44 | 2.571 | Extraction | 'Desk chair for $30.' | 4 (Hallucinated Python code) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 25.74 | 1.942 | Extraction | 'Bake sale, cookies 2$.' | 10 (Excellent Table format) |
| Qwen/Qwen2.5-0.5B-Instruct | Ultra-Light | 25.98 | 1.925 | Extraction | 'Tutoring, $15...' | 9 (Extracted value correctly) |
| HuggingFaceTB/SmolLM2-360M | Ultra-Light | 19.00 | 2.632 | Classification | 'Hello! The Illini Service Dogs...' | 3 (Failed to classify) |
| HuggingFaceTB/SmolLM2-360M | Ultra-Light | 17.65 | 2.833 | Classification | 'Offering math tutoring...' | 4 (Vague explanation) |
| HuggingFaceTB/SmolLM2-360M | Ultra-Light | 21.29 | 0.799 | Classification | 'Selling a used air fryer...' | 2 (Asked a question back) |
| HuggingFaceTB/SmolLM2-360M | Ultra-Light | 21.22 | 2.356 | Classification | 'Car wash event...' | 4 (Analyzed grammar only) |
| HuggingFaceTB/SmolLM2-360M | Ultra-Light | 21.24 | 1.318

In [ ]:
import torch
import time
import json
import gc
import os
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- 1. CONFIGURATION ---
MODEL_MAP = {
    "Medium": [
       "Qwen/Qwen2.5-7B-Instruct",
       "Deci/DeciLM-7B"
    ]
}

prompts_data = [
    {"type": "Classification", "text": "Classify: 'Hello! The Illini Service Dogs are having a fundraiser bake sale today.'"},
    {"type": "Classification", "text": "Classify: 'Offering math tutoring services for $20 an hour.'"},
    {"type": "Classification", "text": "Classify: 'Selling a used air fryer for $30'"},
    {"type": "Classification", "text": "Classify: 'Car wash event to raise money for a student organization.'"},
    {"type": "Classification", "text": "Classify: 'Freelance photography, graduation photoshoot available.'"},
    {"type": "Prediction", "text": "Does this violate policy? Offering essay writing services for payment."},
    {"type": "Prediction", "text": "Determine if safe: Selling used textbooks."},
    {"type": "Prediction", "text": "Is this listing appropriate? Providing tutoring help for exams."},
    {"type": "Prediction", "text": "Check policy compliance: Selling stolen electronics."},
    {"type": "Prediction", "text": "Does this violate university policy? Promoting a local bar crawl."},
    {"type": "Summarize", "text": "Summarize into a short tagline: 'Used bike in good condition, recently repaired, perfect for campus travel.'"},
    {"type": "Summarize", "text": "Create a one-line summary: 'Affordable tutoring services for math and science courses.'"},
    {"type": "Summarize", "text": "Summarize this listing: 'Mini fridge with freezer, lightly used, great for dorm rooms.'"},
    {"type": "Summarize", "text": "Write a short tagline: 'Student organization fundraiser selling homemade cookies.'"},
    {"type": "Summarize", "text": "Summarize: 'Graphic design services for resumes, flyers, and social media.'"},
    {"type": "Text Gen", "text": "Write a message to the seller asking if the item is still available."},
    {"type": "Text Gen", "text": "Generate a polite inquiry about price negotiation."},
    {"type": "Text Gen", "text": "Write a message asking for more details about the service."},
    {"type": "Text Gen", "text": "Generate a friendly message expressing interest in buying."},
    {"type": "Text Gen", "text": "Write a short message scheduling a meetup."},
    {"type": "Extraction", "text": "Extract fields: 'Mini fridge, $60, Urbana campus.'"},
    {"type": "Extraction", "text": "Convert to structured format: 'Graphic design service, $15 per project, remote.'"},
    {"type": "Extraction", "text": "Extract {item, price}: 'Desk chair for $30.'"},
    {"type": "Extraction", "text": "Convert to structured format: 'Bake sale at the Quad, cookies 2$ each.'"},
    {"type": "Extraction", "text": "Extract fields: 'Tutoring, $15, Urbana campus.'"}
]

OUTPUT_FILE = "A7_Model_Benchmark_Results_Medium(2).json"

# --- DEVICE SETUP FOR MAC M2 ---
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Targeting device: APPLE SILICON (MPS)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Targeting device: NVIDIA (CUDA)")
else:
    device = torch.device("cpu")
    print("Targeting device: CPU (Warning: Performance will be slow)")

# --- 2. EXECUTION LOOP ---
for category, models in MODEL_MAP.items():
    for model_id in models:
        print(f"\n{'='*50}\n>>> PROCESSING: {model_id} ({category})\n{'='*50}")

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            # Using float16 is vital for M2 memory and speed
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                dtype=torch.float16,
                trust_remote_code=True,
                low_cpu_mem_usage=True
            ).to(device)

            if model.config.pad_token_id is None:
                model.config.pad_token_id = model.config.eos_token_id

            all_benchmarks = [] # Reset for this specific model

            for p in prompts_data:
                inputs = tokenizer(p['text'], return_tensors="pt").to(device)

                start_time = time.perf_counter()
                # Generation
                output_tokens = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    do_sample=True,    # Now temperature/top_p will be active
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=tokenizer.pad_token_id
                )
                end_time = time.perf_counter()

                latency = end_time - start_time
                new_tokens = len(output_tokens[0]) - len(inputs['input_ids'][0])
                tps = new_tokens / latency if latency > 0 else 0

                response_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

                all_benchmarks.append({
                    "Category": category,
                    "Model": model_id,
                    "Task Type": p['type'],
                    "Prompt": p['text'],
                    "Result": response_text,
                    "Latency (sec)": round(latency, 3),
                    "Tokens Per Sec": round(tps, 2)
                })
                print(f"  [+] {p['type']} completed. Speed: {tps:.2f} t/s")

            # --- INCREMENTAL SAVE ---
            final_data = []
            if os.path.exists(OUTPUT_FILE):
                with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
                    try:
                        final_data = json.load(f)
                    except json.JSONDecodeError:
                        final_data = []

            final_data.extend(all_benchmarks)

            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(final_data, f, indent=4)

            print(f"\n[INFO] Progress for {model_id} appended to {OUTPUT_FILE}")

        except Exception as e:
            print(f"\n[ERROR] Skipping {model_id} due to failure: {e}")
            continue

        # --- 3. MEMORY FLUSH (Apple Silicon specific) ---
        del model
        del tokenizer
        # There is no mps.empty_cache() exactly like cuda, but this helps:
        gc.collect()
        time.sleep(2)

print("\n" + "#"*50)
print(f"BENCHMARK FINISHED. Final JSON path: {os.path.abspath(OUTPUT_FILE)}")
print("#"*50)

Targeting device: APPLE SILICON (MPS)

>>> PROCESSING: Qwen/Qwen2.5-7B-Instruct (Medium)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]